In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

# Defining Model

In [4]:
#model = 'XGBoost'
#model = 'RandomForest'
model = 'TabPFN'

# Physical Calculation (Pullout)

In [5]:
def compute_physical_calc(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

x_train['Physical_Calc'] = compute_physical_calc(x_train) 
x_dev['Physical_Calc'] = compute_physical_calc(x_dev)

y_train_delta = y_train - x_train['Physical_Calc'] 
y_dev_delta = y_dev - x_dev['Physical_Calc']

x_train_res = x_train.drop(columns=['Physical_Calc'])
x_dev_res = x_dev.drop(columns=['Physical_Calc'])

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(X=x_dev)

In [6]:
print(x_dev['Physical_Calc'])


0     5967.0
1     6010.2
2     6042.8
3     6064.5
4     6097.2
       ...  
94    3153.9
95    3145.6
96    3170.5
97    3112.5
98    3153.9
Name: Physical_Calc, Length: 99, dtype: float64


In [7]:
print(y_dev)

0     4161.4
1     1836.4
2     3970.1
3     2509.8
4     4952.0
       ...  
94    2865.5
95    2860.7
96    2902.6
97    3000.7
98    2054.5
Name: PullTest (N), Length: 99, dtype: float64


# Fit Model

In [8]:
if model == 'TabPFN':
    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train_res, y_train_delta)

    # Predict on the test set
    predictions = regressor.predict(x_dev_res)

elif model == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train_delta)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev_delta)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 5,
        'eta': 0.3,
        'eval_metric': 'rmse'
    }

    # Train the model
    num_boost_round = 10
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)

elif model == 'RandomForest':

    # Set the parameters for the Random Forest model
    params = {
            'n_estimators': 11,
            'max_depth': 5,
            'min_samples_split': 2,
            'min_samples_leaf': 4,
            'random_state': 42,
    }

    predictions = RandomForestRegressor(**params).fit(x_train_scale, y_train_delta).predict(x_dev_scale)

final_pred = x_dev['Physical_Calc'] + predictions

# Check Validation Data

In [9]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=final_pred,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model}) by Category"
)

# Check Validation Loss and R2

In [10]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, final_pred)
rmse = root_mean_squared_error(y_dev, final_pred)
R2   = r2_score(y_dev, final_pred)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  129.04
RMSE: 216.69
R2: 0.63


# Cross Validation

In [11]:
cross_df = pd.read_csv("data/train_dev_data.csv")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

mae_list = []
rmse_list = []
R2_list = []

for fold, (train_index, val_index) in enumerate(skf.split(cross_df["Sample ID"], cross_df["Category"])):
    x_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column) 
    x_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)


    x_tr['Physical_Calc'] = compute_physical_calc(x_tr) 
    x_val['Physical_Calc'] = compute_physical_calc(x_val)

    y_tr_delta = y_tr - x_tr['Physical_Calc'] 
    y_val_delta = y_val - x_val['Physical_Calc']

    x_train_res = x_tr.drop(columns=['Physical_Calc'])
    x_dev_res = x_val.drop(columns=['Physical_Calc'])

    regressor = TabPFNRegressor()
    regressor.fit(x_train_res, y_tr_delta)

    preds = regressor.predict(x_dev_res)
    final_pred = x_val['Physical_Calc'] + preds

    mae  = mean_absolute_error(y_val, final_pred)
    rmse = root_mean_squared_error(y_val, final_pred)
    R2   = r2_score(y_val, final_pred)

    mae_list.append(mae) 
    rmse_list.append(rmse) 
    R2_list.append(R2)

    # Categories
    categories= cross_df.iloc[val_index]["Category"].values

    plot_visualizer(
        true_vals=y_val,
        pred_vals=final_pred,
        categories=categories,
        title=f"Fold {fold+1}: True vs Prediction ({model}) by Category"
    )

    print(f"\nFold {fold+1}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", R2)

mae_mean = np.mean(mae_list) 
rmse_mean = np.mean(rmse_list) 
R2_mean = np.mean(R2_list) 


Fold 1
MAE : 135.86487642346006
RMSE: 238.2621840350543
R²  : 0.6083562390320651



Fold 2
MAE : 136.45620037690372
RMSE: 257.38060270772337
R²  : 0.696775962150107



Fold 3
MAE : 128.08675202813768
RMSE: 199.42241204173862
R²  : 0.7290374201685981


In [12]:
# Final Evaluation over all Folds
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")


mean MAE:  133.47
mean RMSE: 231.69
mean R²:   0.68
